# 🚀 Complete X/Twitter Scraper & Analyzer

**One-Stop Solution for X/Twitter Data Collection and Analysis**

This notebook provides everything you need to:
- ✅ Scrape X/Twitter data (tweets, users, followers)
- 📊 Convert data to pandas DataFrames
- 📈 Analyze and visualize your results
- 💾 Export data in multiple formats

**⚠️ Important Notes:**
- Twitter scraping requires accounts to be configured
- Some features may be rate-limited
- Respect Twitter's terms of service and robots.txt
- Use responsibly and ethically

---

## 📦 1. Setup & Dependencies

Install required packages and set up the environment.

In [ ]:
# Install required packages
!pip install twscrape pandas matplotlib seaborn nest-asyncio plotly

# Import libraries
import asyncio
import json
import sys
from pathlib import Path
from typing import List, Dict, Any, Optional, Union
from contextlib import aclosing
import nest_asyncio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

# Enable nested event loops for Colab
nest_asyncio.apply()

# Set up plotting styles
plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Import twscrape
from twscrape import API, gather
from twscrape.logger import set_log_level

# Set log level
set_log_level("INFO")

# Global variables for data storage
scraped_data = {}
dataframes = {}

print("✅ Complete setup finished!")
print("🔧 Libraries loaded and configured")
print("📊 Ready for data collection and analysis")

## 🔑 2. Account Management

Configure your Twitter accounts for scraping.

In [ ]:
class TwitterScraper:
    def __init__(self):
        self.api = API()

    async def check_accounts(self):
        """Check available accounts"""
        try:
            accounts = await self.api.pool.get_all()
            if not accounts:
                return False, "No accounts configured"

            active_accounts = [acc for acc in accounts if acc.active]
            inactive_accounts = [acc for acc in accounts if not acc.active]

            return len(active_accounts) > 0, f"{len(active_accounts)} active accounts available"
        except Exception as e:
            return False, f"Error checking accounts: {e}"

    async def search_tweets(self, query: str, limit: int = 100) -> List[Dict[str, Any]]:
        """Search for tweets using a query"""
        tweets = []
        try:
            async with aclosing(self.api.search(query, limit=limit)) as gen:
                async for tweet in gen:
                    tweet_data = {
                        'id': tweet.id,
                        'url': tweet.url,
                        'date': tweet.date.isoformat() if tweet.date else None,
                        'content': tweet.rawContent,
                        'username': tweet.user.username,
                        'display_name': tweet.user.displayname,
                        'user_id': tweet.user.id,
                        'verified': getattr(tweet.user, 'verified', False),
                        'followers_count': getattr(tweet.user, 'followersCount', 0),
                        'following_count': getattr(tweet.user, 'followingCount', getattr(tweet.user, 'friendsCount', 0)),
                        'reply_count': getattr(tweet, 'replyCount', 0),
                        'retweet_count': getattr(tweet, 'retweetCount', 0),
                        'like_count': getattr(tweet, 'likeCount', 0),
                        'quote_count': getattr(tweet, 'quoteCount', 0),
                        'view_count': getattr(tweet, 'viewCount', 0),
                        'lang': getattr(tweet, 'lang', ''),
                        'source': getattr(tweet, 'sourceLabel', ''),
                        'hashtags': [getattr(tag, 'text', str(tag)) for tag in getattr(tweet, 'hashtags', []) or []],
                        'mentioned_users': [user.username for user in getattr(tweet, 'mentionedUsers', []) or []],
                        'links': [{
                            'text': getattr(link, 'text', ''),
                            'url': getattr(link, 'url', ''),
                            'type': getattr(link, 'type', 'unknown')
                        } for link in getattr(tweet, 'links', []) or []],
                        'media': [{
                            'type': getattr(media, 'type', 'unknown'),
                            'url': getattr(media, 'url', ''),
                            'alt': getattr(media, 'alt', '')
                        } for media in (tweet.media if isinstance(tweet.media, list) else [tweet.media] if tweet.media else [])]
                    }
                    tweets.append(tweet_data)
                    if len(tweets) % 10 == 0:
                        print(f"📝 Collected {len(tweets)}/{limit} tweets")
        except Exception as e:
            print(f"❌ Error searching tweets: {e}")

        return tweets

    async def get_user_info(self, username: str) -> Dict[str, Any]:
        """Get detailed user information"""
        try:
            user = await self.api.user_by_login(username)
            return {
                'id': user.id,
                'username': user.username,
                'display_name': user.displayname,
                'description': getattr(user, 'rawDescription', ''),
                'verified': getattr(user, 'verified', False),
                'protected': getattr(user, 'protected', False),
                'followers_count': getattr(user, 'followersCount', 0),
                'following_count': getattr(user, 'followingCount', getattr(user, 'friendsCount', 0)),
                'statuses_count': getattr(user, 'statusesCount', 0),
                'favourites_count': getattr(user, 'favouritesCount', 0),
                'listed_count': getattr(user, 'listedCount', 0),
                'media_count': getattr(user, 'mediaCount', 0),
                'location': getattr(user, 'location', ''),
                'url': getattr(user, 'url', ''),
                'joined_date': user.created.isoformat() if getattr(user, 'created', None) else None,
                'profile_banner_url': getattr(user, 'profileBannerUrl', ''),
                'profile_image_url': getattr(user, 'profileImageUrl', '')
            }
        except Exception as e:
            print(f"❌ Error getting user info for {username}: {e}")
            return {}

    async def get_user_tweets(self, username: str, limit: int = 50) -> List[Dict[str, Any]]:
        """Get recent tweets from a user"""
        try:
            user = await self.api.user_by_login(username)
            tweets = await gather(self.api.user_tweets(user.id, limit=limit))
            return [{
                'id': tweet.id,
                'url': tweet.url,
                'date': tweet.date.isoformat() if tweet.date else None,
                'content': tweet.rawContent,
                'reply_count': tweet.replyCount,
                'retweet_count': tweet.retweetCount,
                'like_count': tweet.likeCount,
                'view_count': tweet.viewCount
            } for tweet in tweets]
        except Exception as e:
            print(f"❌ Error getting tweets for {username}: {e}")
            return []

    async def get_followers(self, username: str, limit: int = 100) -> List[Dict[str, Any]]:
        """Get followers of a user"""
        try:
            user = await self.api.user_by_login(username)
            followers = await gather(self.api.followers(user.id, limit=limit))
            return [{
                'id': follower.id,
                'username': follower.username,
                'display_name': follower.displayname,
                'verified': getattr(follower, 'verified', False),
                'followers_count': getattr(follower, 'followersCount', 0),
                'following_count': getattr(follower, 'followingCount', getattr(follower, 'friendsCount', 0)),
                'description': getattr(follower, 'rawDescription', '')
            } for follower in followers]
        except Exception as e:
            print(f"❌ Error getting followers for {username}: {e}")
            return []

# Initialize scraper
scraper = TwitterScraper()

# Check account status
accounts_ok, message = await scraper.check_accounts()
print(f"🔍 Account Status: {message}")

if not accounts_ok:
    print("\n📋 To set up accounts:")
    print("1. Create an accounts.txt file with your Twitter credentials")
    print("2. Format: username:password:email:email_password")
    print("3. Upload it to Colab or create it here")
    print("\nExample accounts.txt content:")
    print("your_username:your_password:your_email@gmail.com:email_password")
else:
    print("✅ Ready to scrape!")

### Upload Accounts File (Optional)

In [ ]:
# Upload accounts file
from google.colab import files

try:
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        print(f'✅ Uploaded: {filename}')
        if filename == 'accounts.txt':
            !mv {filename} .
            print("✅ Accounts file ready!")
            break
except:
    print("⚠️ File upload not available or cancelled")

### Add Account Manually (if needed)

In [ ]:
#@title Add Twitter Account

# Uncomment and fill in your credentials
# username = "your_username"  #@param {type:"string"}
# password = "your_password"  #@param {type:"string"}
# email = "your_email@gmail.com"  #@param {type:"string"}
# email_password = "email_password"  #@param {type:"string"}

# If you filled in the credentials above, uncomment the next lines:
# await scraper.api.pool.add_account(username, password, email, email_password)
# await scraper.api.pool.login_all()
# print("✅ Account added and logged in!")

print("⚠️  Please uncomment and fill in your credentials above, then run this cell.")

## 🔍 3. Data Collection

Choose what type of data you want to collect:

In [ ]:
#@title 🔍 Search Tweets

search_query = "AI technology"  #@param {type:"string"}
search_limit = 50  #@param {type:"slider", min:10, max:500, step:10}
run_search = False  #@param {type:"boolean"}
save_to_memory = True  #@param {type:"boolean"}

if run_search and search_query.strip():
    print(f"🔍 Searching for: '{search_query}' (limit: {search_limit})")
    tweets = await scraper.search_tweets(search_query, search_limit)
    
    if tweets:
        # Save to memory
        data_key = f"search_{search_query.replace(' ', '_')}"
        scraped_data[data_key] = tweets
        print(f"✅ Collected {len(tweets)} tweets")
        
        if save_to_memory:
            # Convert to DataFrame immediately
            df = convert_tweets_to_dataframe(tweets)
            dataframes[data_key] = df
            print(f"📊 DataFrame created: {data_key}")
            
        # Save to JSON file
        filename = f"{data_key}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(tweets, f, indent=2, ensure_ascii=False)
        print(f"💾 Saved to file: {filename}")
        
    else:
        print("❌ No tweets found")
else:
    print("⚙️ Configure search parameters above and set 'run_search' to True")

In [ ]:
#@title 👤 Get User Information

user_username = "elonmusk"  #@param {type:"string"}
run_user_info = False  #@param {type:"boolean"}
save_to_memory = True  #@param {type:"boolean"}

if run_user_info and user_username.strip():
    print(f"👤 Getting info for: @{user_username}")
    user_info = await scraper.get_user_info(user_username)
    
    if user_info:
        # Save to memory
        data_key = f"user_{user_username}"
        scraped_data[data_key] = user_info
        print(f"✅ User info collected")
        
        if save_to_memory:
            # Convert to DataFrame immediately
            df = convert_user_info_to_dataframe(user_info)
            dataframes[data_key] = df
            print(f"📊 DataFrame created: {data_key}")
            
        # Save to JSON file
        filename = f"{data_key}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(user_info, f, indent=2, ensure_ascii=False)
        print(f"💾 Saved to file: {filename}")
        
    else:
        print("❌ User not found or error occurred")
else:
    print("⚙️ Enter a username above and set 'run_user_info' to True")

In [ ]:
#@title 🐦 Get User Tweets

tweets_username = "elonmusk"  #@param {type:"string"}
tweets_limit = 25  #@param {type:"slider", min:5, max:200, step:5}
run_user_tweets = False  #@param {type:"boolean"}
save_to_memory = True  #@param {type:"boolean"}

if run_user_tweets and tweets_username.strip():
    print(f"🐦 Getting tweets for: @{tweets_username} (limit: {tweets_limit})")
    tweets = await scraper.get_user_tweets(tweets_username, tweets_limit)
    
    if tweets:
        # Save to memory
        data_key = f"tweets_{tweets_username}"
        scraped_data[data_key] = tweets
        print(f"✅ Collected {len(tweets)} tweets")
        
        if save_to_memory:
            # Convert to DataFrame immediately
            df = convert_tweets_to_dataframe(tweets)
            dataframes[data_key] = df
            print(f"📊 DataFrame created: {data_key}")
            
        # Save to JSON file
        filename = f"{data_key}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(tweets, f, indent=2, ensure_ascii=False)
        print(f"💾 Saved to file: {filename}")
        
    else:
        print("❌ No tweets found or error occurred")
else:
    print("⚙️ Enter a username above and set 'run_user_tweets' to True")

In [ ]:
#@title 👥 Get User Followers

followers_username = "elonmusk"  #@param {type:"string"}
followers_limit = 50  #@param {type:"slider", min:10, max:500, step:10}
run_followers = False  #@param {type:"boolean"}
save_to_memory = True  #@param {type:"boolean"}

if run_followers and followers_username.strip():
    print(f"👥 Getting followers for: @{followers_username} (limit: {followers_limit})")
    followers = await scraper.get_followers(followers_username, followers_limit)
    
    if followers:
        # Save to memory
        data_key = f"followers_{followers_username}"
        scraped_data[data_key] = followers
        print(f"✅ Collected {len(followers)} followers")
        
        if save_to_memory:
            # Convert to DataFrame immediately
            df = convert_followers_to_dataframe(followers)
            dataframes[data_key] = df
            print(f"📊 DataFrame created: {data_key}")
            
        # Save to JSON file
        filename = f"{data_key}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(followers, f, indent=2, ensure_ascii=False)
        print(f"💾 Saved to file: {filename}")
        
    else:
        print("❌ No followers found or error occurred")
else:
    print("⚙️ Enter a username above and set 'run_followers' to True")

## 📊 4. Data Processing & Analysis

Convert your scraped data to pandas DataFrames and analyze it.

In [ ]:
# Data conversion functions

def convert_tweets_to_dataframe(tweets_data: List[Dict]) -> pd.DataFrame:
    """Convert tweets data to a pandas DataFrame"""
    if not tweets_data:
        return pd.DataFrame()
    
    df = pd.DataFrame(tweets_data)
    
    # Convert date strings to datetime
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # Flatten nested structures
    if 'hashtags' in df.columns:
        df['hashtags_count'] = df['hashtags'].apply(lambda x: len(x) if isinstance(x, list) else 0)
        df['hashtags_str'] = df['hashtags'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')
    
    if 'mentioned_users' in df.columns:
        df['mentioned_users_count'] = df['mentioned_users'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    
    if 'media' in df.columns:
        df['media_count'] = df['media'].apply(lambda x: len(x) if isinstance(x, list) else 0)
        df['has_media'] = df['media_count'] > 0
    
    if 'links' in df.columns:
        df['links_count'] = df['links'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    
    # Add engagement score
    df['engagement_score'] = (
        df.get('like_count', 0) + 
        df.get('retweet_count', 0) + 
        df.get('reply_count', 0) + 
        df.get('quote_count', 0)
    )
    
    # Add content length
    df['content_length'] = df['content'].fillna('').str.len()
    
    return df

def convert_user_info_to_dataframe(user_data: Dict) -> pd.DataFrame:
    """Convert user info data to a pandas DataFrame"""
    if not user_data:
        return pd.DataFrame()
    
    df = pd.DataFrame([user_data])
    
    if 'joined_date' in df.columns:
        df['joined_date'] = pd.to_datetime(df['joined_date'], errors='coerce')
    
    df['account_age_days'] = (pd.Timestamp.now() - df['joined_date']).dt.days
    df['follower_following_ratio'] = df['followers_count'] / df['following_count'].replace(0, 1)
    df['statuses_per_day'] = df['statuses_count'] / df['account_age_days'].replace(0, 1)
    
    return df

def convert_followers_to_dataframe(followers_data: List[Dict]) -> pd.DataFrame:
    """Convert followers data to a pandas DataFrame"""
    if not followers_data:
        return pd.DataFrame()
    
    df = pd.DataFrame(followers_data)
    df['follower_following_ratio'] = df['followers_count'] / df['following_count'].replace(0, 1)
    df['description_length'] = df['description'].fillna('').str.len()
    
    return df

# Process any scraped data that wasn't converted yet
for key, data in scraped_data.items():
    if key not in dataframes:
        if key.startswith('search_') or key.startswith('tweets_'):
            dataframes[key] = convert_tweets_to_dataframe(data)
        elif key.startswith('user_'):
            dataframes[key] = convert_user_info_to_dataframe(data)
        elif key.startswith('followers_'):
            dataframes[key] = convert_followers_to_dataframe(data)

print(f"📊 Available DataFrames: {list(dataframes.keys())}")
print(f"📦 Scraped data in memory: {list(scraped_data.keys())}")

### Upload Existing JSON Files (Optional)

In [ ]:
# Upload and convert existing JSON files
try:
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        if filename.endswith('.json'):
            print(f"📄 Processing uploaded file: {filename}")
            
            with open(filename, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Detect data type and convert
            data_key = Path(filename).stem
            
            if isinstance(data, list) and data:
                if 'content' in data[0] or 'rawContent' in data[0]:
                    df = convert_tweets_to_dataframe(data)
                else:
                    df = convert_followers_to_dataframe(data)
            elif isinstance(data, dict):
                df = convert_user_info_to_dataframe(data)
            else:
                df = pd.DataFrame()
            
            if not df.empty:
                dataframes[data_key] = df
                print(f"✅ DataFrame created: {data_key} (shape: {df.shape})")
            else:
                print(f"⚠️ Could not convert {filename}")
        
except Exception as e:
    print(f"⚠️ File upload error: {e}")

print(f"\n📊 Total DataFrames available: {len(dataframes)}")
for name, df in dataframes.items():
    print(f"  - {name}: {df.shape}")

### Data Overview

In [ ]:
#@title 📊 Data Overview

selected_dataframe = ""  #@param {type:"string"}
show_preview = True  #@param {type:"boolean"}
show_info = True  #@param {type:"boolean"}
show_stats = True  #@param {type:"boolean"}

if not selected_dataframe and dataframes:
    selected_dataframe = list(dataframes.keys())[0]

if selected_dataframe in dataframes:
    df = dataframes[selected_dataframe]
    
    print(f"📊 Analyzing: {selected_dataframe}")
    print(f"Shape: {df.shape}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    
    if show_info:
        print("\n📋 Data Types:")
        print(df.dtypes)
        
        print("\n❓ Missing Values:")
        missing = df.isnull().sum()
        if missing.sum() > 0:
            print(missing[missing > 0])
        else:
            print("No missing values found")
    
    if show_preview:
        print("\n🔍 Preview:")
        display(df.head())
    
    if show_stats:
        print("\n📈 Basic Statistics:")
        display(df.describe())
        
else:
    print("Available DataFrames:")
    for name in dataframes.keys():
        print(f"  - {name}")
    print("\nSelect one from the dropdown above")

## 📈 5. Data Visualization & Insights

Explore your data with interactive charts and analysis.

In [ ]:
#@title 📊 Interactive Analysis

analyze_dataframe = ""  #@param {type:"string"}
analysis_type = "overview"  #@param ["overview", "engagement", "temporal", "network", "content"]

if not analyze_dataframe and dataframes:
    analyze_dataframe = list(dataframes.keys())[0]

if analyze_dataframe in dataframes:
    df = dataframes[analyze_dataframe]
    
    if analysis_type == "overview":
        # Basic overview
        print(f"📊 {analyze_dataframe} Overview")
        print(f"Records: {len(df)}")
        
        # Show numeric columns distribution
        numeric_cols = df.select_dtypes(include=["number"]).columns
        if len(numeric_cols) > 0:
            fig, axes = plt.subplots(2, 2, figsize=(15, 10))
            fig.suptitle(f'{analyze_dataframe} - Numeric Distributions')
            
            for i, col in enumerate(numeric_cols[:4]):
                ax = axes[i//2, i%2]
                df[col].hist(bins=30, ax=ax)
                ax.set_title(col)
                ax.set_xlabel(col)
                ax.set_ylabel('Frequency')
            
            plt.tight_layout()
            plt.show()
    
    elif analysis_type == "engagement" and ('engagement_score' in df.columns or 'like_count' in df.columns):
        # Engagement analysis
        print("🚀 Engagement Analysis")
        
        if 'engagement_score' in df.columns:
            # Top tweets by engagement
            top_tweets = df.nlargest(10, 'engagement_score')
            print("\n🏆 Top 10 Tweets by Engagement:")
            display(top_tweets[['username', 'content', 'engagement_score', 'like_count', 'retweet_count']].head())
            
            # Engagement distribution
            fig = px.histogram(df, x='engagement_score', title='Engagement Score Distribution',
                             nbins=50, marginal='box')
            fig.show()
        
        # Correlation heatmap
        engagement_cols = [col for col in ['like_count', 'retweet_count', 'reply_count', 'quote_count', 'view_count'] if col in df.columns]
        if len(engagement_cols) > 1:
            corr_matrix = df[engagement_cols].corr()
            fig = px.imshow(corr_matrix, title='Engagement Metrics Correlation',
                          text_auto='.2f', aspect='auto')
            fig.show()
    
    elif analysis_type == "temporal" and 'date' in df.columns:
        # Time-based analysis
        print("⏰ Temporal Analysis")
        
        # Convert to datetime if needed
        df_time = df.copy()
        df_time['date'] = pd.to_datetime(df_time['date'], errors='coerce')
        df_time = df_time.dropna(subset=['date'])
        
        if not df_time.empty:
            # Group by date
            daily_stats = df_time.set_index('date').resample('D').agg({
                'engagement_score': 'sum',
                'id': 'count'
            }).rename(columns={'id': 'tweet_count'})
            
            # Plot time series
            fig = make_subplots(specs=[[{"secondary_y": True}]])
            fig.add_trace(go.Scatter(x=daily_stats.index, y=daily_stats['tweet_count'], 
                                   name="Tweet Count", mode='lines+markers'))
            if 'engagement_score' in daily_stats.columns:
                fig.add_trace(go.Scatter(x=daily_stats.index, y=daily_stats['engagement_score'], 
                                       name="Engagement", mode='lines+markers'), secondary_y=True)
            fig.update_layout(title='Tweets Over Time', xaxis_title='Date')
            fig.update_yaxes(title_text="Tweet Count", secondary_y=False)
            fig.update_yaxes(title_text="Engagement Score", secondary_y=True)
            fig.show()
    
    elif analysis_type == "content" and 'content' in df.columns:
        # Content analysis
        print("📝 Content Analysis")
        
        # Content length distribution
        fig = px.histogram(df, x='content_length', title='Tweet Length Distribution',
                         nbins=50, marginal='box')
        fig.show()
        
        # Hashtag analysis
        if 'hashtags' in df.columns:
            all_hashtags = []
            for hashtags in df['hashtags'].dropna():
                if isinstance(hashtags, list):
                    all_hashtags.extend(hashtags)
            
            if all_hashtags:
                hashtag_counts = pd.Series(all_hashtags).value_counts().head(20)
                fig = px.bar(hashtag_counts, title='Top 20 Hashtags',
                           labels={'index': 'Hashtag', 'value': 'Count'})
                fig.show()
    
    elif analysis_type == "network" and 'username' in df.columns:
        # Network analysis
        print("🌐 Network Analysis")
        
        if 'username' in df.columns:
            # Top users by frequency
            user_counts = df['username'].value_counts().head(20)
            fig = px.bar(user_counts, title='Most Active Users',
                       labels={'index': 'Username', 'value': 'Tweet Count'})
            fig.update_xaxes(tickangle=45)
            fig.show()
            
            # Verified users analysis
            if 'verified' in df.columns:
                verified_stats = df.groupby('verified').agg({
                    'engagement_score': 'mean',
                    'followers_count': 'mean',
                    'username': 'count'
                }).round(2)
                print("\n✅ Verified vs Non-Verified Users:")
                display(verified_stats)
    
else:
    print("❌ No data available. Please scrape some data first or upload JSON files.")
    print("Available DataFrames:", list(dataframes.keys()))

### Custom Analysis Code

Add your own analysis code here:

In [ ]:
# Custom analysis space
# Available DataFrames: list(dataframes.keys())

# Example: Find tweets with highest engagement
# if 'search_AI_technology' in dataframes:
#     df = dataframes['search_AI_technology']
#     top_tweets = df.nlargest(5, 'engagement_score')
#     display(top_tweets[['username', 'content', 'engagement_score']])

# Example: Analyze follower demographics
# if 'followers_elonmusk' in dataframes:
#     df = dataframes['followers_elonmusk']
#     print(f"Average followers: {df['followers_count'].mean():.0f}")
#     print(f"Verified followers: {df['verified'].sum()}")

print("💡 Add your custom analysis code here!")
print("Available DataFrames:", list(dataframes.keys()))
print("\nExample:")
print("df = dataframes['your_dataframe_name']")
print("df.describe()")

## 💾 6. Export & Download

Save your results and download them.

In [ ]:
#@title 💾 Export Data

export_format = "csv"  #@param ["csv", "json", "excel"]
export_all_dataframes = True  #@param {type:"boolean"}
specific_dataframe = ""  #@param {type:"string"}
include_visualizations = False  #@param {type:"boolean"}

from google.colab import files
import io

exported_files = []

if dataframes:
    dataframes_to_export = list(dataframes.keys()) if export_all_dataframes else [specific_dataframe]
    
    for df_name in dataframes_to_export:
        if df_name in dataframes:
            df = dataframes[df_name]
            
            if export_format == "csv":
                filename = f"{df_name}.csv"
                df.to_csv(filename, index=False, encoding='utf-8')
                files.download(filename)
                exported_files.append(filename)
                
            elif export_format == "json":
                filename = f"{df_name}.json"
                df.to_json(filename, orient='records', indent=2, date_format='iso')
                files.download(filename)
                exported_files.append(filename)
                
            elif export_format == "excel":
                filename = f"{df_name}.xlsx"
                df.to_excel(filename, index=False, engine='openpyxl')
                files.download(filename)
                exported_files.append(filename)
    
    print(f"✅ Exported {len(exported_files)} files in {export_format.upper()} format")
    
    # Export raw JSON data if available
    if scraped_data:
        for data_name, data in scraped_data.items():
            filename = f"raw_{data_name}.json"
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False, default=str)
            files.download(filename)
            exported_files.append(filename)
        print(f"✅ Exported {len(scraped_data)} raw JSON files")
    
    # Export summary report
    summary_data = {
        'export_timestamp': str(pd.Timestamp.now()),
        'dataframes': {name: {'shape': df.shape, 'columns': list(df.columns)} for name, df in dataframes.items()},
        'exported_files': exported_files
    }
    
    summary_filename = "export_summary.json"
    with open(summary_filename, 'w', encoding='utf-8') as f:
        json.dump(summary_data, f, indent=2, default=str)
    files.download(summary_filename)
    
    print(f"\n📋 Summary exported: {summary_filename}")
    
else:
    print("❌ No data available to export. Please collect some data first.")

print(f"\n📊 Session Summary:")
print(f"   DataFrames created: {len(dataframes)}")
print(f"   Raw data collected: {len(scraped_data)}")
print(f"   Files exported: {len(exported_files) if 'exported_files' in locals() else 0}")

## 📋 Session Summary

Review what you've collected and analyzed in this session:

In [ ]:
# Session summary
print("🎯 X/Twitter Scraper Session Summary")
print("=" * 50)

print(f"📊 DataFrames Available: {len(dataframes)}")
for name, df in dataframes.items():
    print(f"   • {name}: {df.shape[0]} rows × {df.shape[1]} columns")

print(f"\n📦 Raw Data in Memory: {len(scraped_data)}")
for name, data in scraped_data.items():
    if isinstance(data, list):
        print(f"   • {name}: {len(data)} records")
    else:
        print(f"   • {name}: 1 record")

# List files created
import os
files_created = [f for f in os.listdir('.') if f.endswith(('.json', '.csv', '.xlsx'))]
print(f"\n💾 Files Created: {len(files_created)}")
for file in sorted(files_created):
    size = os.path.getsize(file) / 1024
    print(f"   • {file}: {size:.1f} KB")

print(f"\n✅ Session completed at: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n💡 Tips:")
print("   • Use the analysis section to explore your data further")
print("   • Export your data before closing this notebook")
print("   • Raw JSON files contain all original data")
print("   • DataFrames are optimized for analysis")

## 🔄 Reset Session (Optional)

Clear all data and start fresh:

In [ ]:
#@title 🔄 Reset Session

confirm_reset = False  #@param {type:"boolean"}

if confirm_reset:
    # Clear all data
    scraped_data.clear()
    dataframes.clear()
    
    # Remove created files
    import os
    for file in os.listdir('.'):
        if file.endswith(('.json', '.csv', '.xlsx')) and not file.startswith('accounts'):
            try:
                os.remove(file)
                print(f"🗑️ Removed: {file}")
            except:
                pass
    
    print("\n🔄 Session reset complete!")
    print("Ready for new data collection.")
else:
    print("⚠️ Check 'confirm_reset' to clear all data")

---

**🎉 You're all set!**

This notebook provides a complete workflow for:
- 🔑 **Account setup** and management
- 🔍 **Data collection** from X/Twitter
- 📊 **Data processing** and analysis
- 📈 **Interactive visualizations**
- 💾 **Export and download** capabilities

**Next steps:**
1. Set up your Twitter accounts
2. Configure your scraping parameters
3. Run data collection
4. Analyze your results
5. Export your findings

**Need help?** Check the documentation in each section or refer to the README files.

---